In [1]:
!pip install mediapipe opencv-python

  Using cached attrs-25.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
  Using cached cffi-1.17.1-cp311-cp311-win_amd64.whl.metadata (1.6 kB)
  Using cached pycparser-2.22-py3-none-any.whl.metadata (943 bytes)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.4.8-cp311-cp311-win_amd64.whl.metadata (6.3 kB)
  Using cached pillow-11.2.1-cp311-cp311-win_amd64.whl.metadata (9.1 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
   ---------------------------------------- 0.0/51.0 MB ? eta -:--:--
   - -------------------------------------- 2.1/51.0 MB 10.7 MB/s eta 0:00:05
   --- ------------------------------------ 4.5/51.0 MB 10.7 MB/s eta 0:00:05
   ----- ---------------------------------- 6.6/51.0 MB 10.6 MB/s eta 0:00:05
   ------ --------------------------------- 8.4/51.0 MB 10.2 MB/s eta 0:00:05
   ------- -------------------------------- 10.0/51.0 MB 9.5 MB/s e

In [ ]:
import cv2
import mediapipe as mp  # 스켈레톤을 추출하는 라이브러리
import numpy as np
import csv  # csv 저장을 위해 라이브러리 추가
import os   # 파일 경로 관리를 위해 라이브러리 추가

# MediaPipe Pose 모델 초기화
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    min_detection_confidence = 0.5,  # 감지 최소 신뢰도
    min_tracking_confidence = 0.5 # 추적 최소 신뢰도
)

# MediaPipe 그리기 유틸리티 초기화
mp_drawing = mp.solutions.drawing_utils

# 동영상 파일 경로
video_path = "TWICE_ICANTSTOPME.mp4" # 분석할 동영상 파일 경로 입력
cap = cv2.VideoCapture(video_path)

# 출력 파일 이름 설정
output_filename = os.path.splitext(os.path.basename(video_path))[0] + "_skeleton.csv"
# CSV 파일 헤더 준비 (33개 랜드마크 * 4개 좌표)
landmarks = ['class'] + [f'{j}_{i}' for i in mp_pose.PoseLandmark._member_names_ for j in ('x', 'y', 'z', 'v')]

# 동영상 파일이 정상적으로 열렸는지 확인
if not cap.isOpened():
    print(f"오류: '{video_path}' 동영상을 열 수 없습니다.")
    exit()

print("스켈레톤 추출을 시작합니다. 종료하려면 'q' 키를 누르세요.")

with open(output_filename, 'w', newline='') as f:
    csv_writer = csv.writer(f)
    csv_writer.writerow(landmarks)  # 헤더 작성

    print(f"'{output_filename}' 파일에 스켈레톤 데이터 저장을 시작합니다.")
    frame_count = 0
    while cap.isOpened():
        # 동영상에서 프레임 읽기
        success, image = cap.read()

        if not success:
            print("동영상 스트림의 끝에 도달했거나 오류가 발생했습니다.")
            break

        # 성능 향상을 위해 이미지를 읽기 전용으로 표시
        image.flags.writeable = False
        # BGR 이미지를 RGB로 변환
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # MediaPipe Pose를 사용하여 포즈 감지 수행
        results = pose.process(image_rgb)

        # 이미지를 다시 쓰기 가능으로 변경
        image.flags.writeable = True

        # 감지된 스켈레톤(포즈 랜드마크)을 원본 이미지에 그리기
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                image,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec = mp_drawing.DrawingSpec(color = (245, 117, 66), thickness=2, circle_radius=2),
                connection_drawing_spec=mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)
            )

            # 랜드마크 데이터 추출 및 csv 행으로 변환
            try:
                # 'dance' 클래스로 분류 (필요에 따라 변경 가능)
                class_name = "dance"

                # 모든 랜드마크의 x, y, z, v 값을 순서대로 리스트에 담기
                pose_row = list(np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten())

                # 클래스 이름과 랜드마크 데이터를 합쳐서 한 행으로 만듦
                row = [class_name] + pose_row
                
                # csv 파일에 한 행 쓰기
                csv_writer.writerow(row)

            except Exception as e:
                print(f"프레임 {frame_count} 처리 중 오류 발생: {e}")
                pass # 오류 발생 시 해당 프레임 건너뜀

        # 결과 영상 출력
        cv2.imshow('MediaPipe Pose Skeleton', image)

        frame_count += 1
        # 'q' 키를 누르면 루프 종료
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

# 자원 해제
cap.release()
cv2.destroyAllWindows()
pose.close()

print("스켈레톤 추출이 완료되었습니다.")

스켈레톤 추출을 시작합니다. 종료하려면 'q' 키를 누르세요.
스켈레톤 추출이 완료되었습니다.
